# 번역가는 대화에도 능하다 [프로젝트]
## Project: 한국 챗봇 만들기

### 라이브러리 버전을 확인합니다

In [1]:
import sys
!"{sys.executable}" -m pip install -q sentencepiece gensim nltk

In [2]:
import numpy
import pandas
import torch
import nltk
import gensim
import random

print(numpy.__version__)
print(pandas.__version__)
print(torch.__version__)
print(nltk.__version__)
print(gensim.__version__)

2.5.1
3.0.3
2.11.0+cu128
3.9.4
4.4.0


c:\Users\chaej\AppData\Local\Programs\Python\Python313\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "cipher": algorithms.TripleDES,
c:\Users\chaej\AppData\Local\Programs\Python\Python313\Lib\site-packages\paramiko\transport.py:253: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "class": algorithms.TripleDES,


### **Step 1. 데이터 다운로드**

`ChatbotData.csv` 파일을 읽는 방법은 `pandas` 라이브러리를 활용합니다. 읽어 온 데이터에서 질문과 대답에 해당하는 `questions`, `answers` 변수를 만들어 주세요!

* [songys/Chatbot_data](https://github.com/songys/Chatbot_data)

In [3]:
import os, urllib.request

os.makedirs('./data', exist_ok=True)

csv_path = './data/ChatbotData.csv'
if not os.path.exists(csv_path):
    url = 'https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv'
    print('다운로드 중...')
    urllib.request.urlretrieve(url, csv_path)
    print('완료!')

import pandas as pd
df = pd.read_csv(csv_path)
print(df.shape)
df.head()

(11823, 3)


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [4]:
questions = list(df['Q'])
answers   = list(df['A'])

print(f'총 쌍 수: {len(questions)}')
for q, a in zip(questions[:5], answers[:5]):
    print(f'Q: {q}')
    print(f'A: {a}')
    print()

총 쌍 수: 11823
Q: 12시 땡!
A: 하루가 또 가네요.

Q: 1지망 학교 떨어졌어
A: 위로해 드립니다.

Q: 3박4일 놀러가고 싶다
A: 여행은 언제나 좋죠.

Q: 3박4일 정도 놀러가고 싶다
A: 여행은 언제나 좋죠.

Q: PPL 심하네
A: 눈살이 찌푸려지죠.



### **Step 2. 데이터 전처리**

아래 조건을 만족하는 `preprocess_sentence()` 함수를 작성하세요.

1. 입력이 영어일 경우, **모든 소문자**로 변환합니다.
2. 특수문자와 한글, 영어, 그리고 주요 특수문자를 제외하고 **나머지를 공백으로 처리**합니다.

In [5]:
import re

def preprocess_sentence(sentence):
    sentence = str(sentence).lower()
    sentence = re.sub(r'[^가-힣a-z0-9?.!,\s]', ' ', sentence)
    sentence = re.sub(r'([?.!,])', r' \1 ', sentence)
    sentence = re.sub(r'\s+', ' ', sentence)
    return sentence.strip()

for q, a in zip(questions[:3], answers[:3]):
    print('Q:', preprocess_sentence(q))
    print('A:', preprocess_sentence(a))
    print()

Q: 12시 땡 !
A: 하루가 또 가네요 .

Q: 1지망 학교 떨어졌어
A: 위로해 드립니다 .

Q: 3박4일 놀러가고 싶다
A: 여행은 언제나 좋죠 .



### **Step 3. 데이터 토큰화 (SentencePiece BPE)**

SentencePiece는 공백 정보를 `▁`로 보존하므로 `decode` 시 원문을 그대로 복원할 수 있습니다.
Q와 A가 모두 한국어이므로 공유 어휘 사전(shared vocab)을 사용합니다.

In [6]:
import sentencepiece as spm

VOCAB_SIZE   = 8000
corpus_path  = './data/chatbot_corpus.txt'
model_prefix = './data/chatbot_spm'

# 전처리된 문장으로 코퍼스 파일 생성
with open(corpus_path, 'w', encoding='utf-8') as f:
    for sent in questions + answers:
        f.write(preprocess_sentence(sent) + '\n')

spm.SentencePieceTrainer.Train(
    f'--input={corpus_path} '
    f'--model_prefix={model_prefix} '
    f'--vocab_size={VOCAB_SIZE} '
    f'--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 '
    f'--character_coverage=0.9995'
)

tokenizer = spm.SentencePieceProcessor()
tokenizer.Load(model_prefix + '.model')

PAD_ID = tokenizer.pad_id()   # 0
BOS_ID = tokenizer.bos_id()   # 1
EOS_ID = tokenizer.eos_id()   # 2

print(f'vocab size: {tokenizer.get_piece_size()}')
print('예시:', tokenizer.encode_as_pieces('오늘 날씨가 어때요?'))
print('ids :', tokenizer.encode_as_ids('오늘 날씨가 어때요?'))

vocab size: 8000
예시: ['▁오늘', '▁날씨', '가', '▁어때', '요', '?']
ids : [75, 537, 8, 346, 6, 7997]


### **Step 4. Augmentation**

주어진 데이터는 **1만 쌍으로 비교적 적은 편**입니다. **Lexical Substitution을 활용한 증강**을 해 보겠습니다.

* [Kyubyong/wordvectors](https://github.com/Kyubyong/wordvectors)

In [7]:
import pickle
from gensim.models import KeyedVectors
import numpy as np

KO_W2V_PATH = './data/ko.bin'
USE_AUG = False

if os.path.exists(KO_W2V_PATH):
    with open(KO_W2V_PATH, 'rb') as f:
        old_model = pickle.load(f, encoding='latin-1')
    wv = KeyedVectors(vector_size=old_model.syn0.shape[1])
    wv.add_vectors(old_model.index2word, old_model.syn0)
    del old_model
    USE_AUG = True
    print(f'vocab: {len(wv)} words, dim: {wv.vector_size}')
else:
    print(f'Not found: {KO_W2V_PATH}')
    print('Kyubyong/wordvectors ko.bin -> ./data/')

vocab: 30185 words, dim: 200


In [ ]:
def lexical_sub(sentence, wv, top_n=5, sub_ratio=0.2):
    tokens = sentence.split()
    new_tokens = []
    for token in tokens:
        if random.random() < sub_ratio:
            try:
                candidates = [w for w, _ in wv.most_similar(token, topn=top_n)]
                new_tokens.append(random.choice(candidates))
            except KeyError:
                new_tokens.append(token)
        else:
            new_tokens.append(token)
    return ' '.join(new_tokens)

# augmentation은 전처리된 원문(어절) 수준에서 수행
pre_questions = [preprocess_sentence(q) for q in questions]
pre_answers   = [preprocess_sentence(a) for a in answers]

aug_questions = list(pre_questions)
aug_answers   = list(pre_answers)

N_AUG = 1
if USE_AUG:
    from tqdm.notebook import tqdm
    for q, a in tqdm(zip(pre_questions, pre_answers),
                      total=len(pre_questions), desc='Augmenting'):
        for _ in range(N_AUG):
            new_q = lexical_sub(q, wv)
            if new_q != q and new_q not in aug_questions:
                aug_questions.append(new_q)
                aug_answers.append(a)

print(f'원본: {len(pre_questions)}, 증강 후: {len(aug_questions)}')

Augmenting:   0%|          | 0/11823 [00:00<?, ?it/s]

원본: 11823, 증강 후: 20806


### **Step 5. 데이터 정규화**

타겟 시퀀스에 `<start>`(BOS)와 `<end>`(EOS) 토큰을 추가합니다.
SentencePiece의 `bos_id`/`eos_id`를 사용합니다.

In [9]:
MAX_LEN = 40

def encode_and_pad(sentences, tokenizer, max_len, add_bos_eos=False):
    result = []
    for sent in sentences:
        ids = tokenizer.encode_as_ids(sent)
        if add_bos_eos:
            ids = [BOS_ID] + ids + [EOS_ID]
        ids = ids[:max_len] + [PAD_ID] * max(0, max_len - len(ids))
        result.append(ids)
    return result

# MAX_LEN 이하 쌍만 필터링
src_sents, tgt_sents = [], []
for q, a in zip(aug_questions, aug_answers):
    q_ids = tokenizer.encode_as_ids(q)
    a_ids = tokenizer.encode_as_ids(a)
    if len(q_ids) <= MAX_LEN and len(a_ids) <= MAX_LEN:
        src_sents.append(q)
        tgt_sents.append(a)

import torch
import numpy as np

enc_data = encode_and_pad(src_sents, tokenizer, MAX_LEN)
dec_data = encode_and_pad(tgt_sents, tokenizer, MAX_LEN + 2, add_bos_eos=True)

enc_tensor = torch.tensor(enc_data, dtype=torch.long)
dec_tensor = torch.tensor(dec_data, dtype=torch.long)

print('enc_tensor:', enc_tensor.shape)
print('dec_tensor:', dec_tensor.shape)

enc_tensor: torch.Size([20806, 40])
dec_tensor: torch.Size([20806, 42])


In [10]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 64

train_dataset    = TensorDataset(enc_tensor, dec_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
print(f'배치 수: {len(train_dataloader)}')

배치 수: 326


### **Step 6. 학습하기**

앞서 학습한 것과 같은 `Transformer` 를 그대로 사용하시면 됩니다!

In [11]:
device = torch.device('cuda:1' if torch.cuda.device_count() > 1 else
                      'cuda'   if torch.cuda.is_available()        else
                      'cpu')
print('device:', device)

device: cuda:1


In [12]:
import torch.nn as nn
import torch.nn.functional as F
import math

def positional_encoding(pos, d_model):
    table = np.zeros((pos, d_model))
    for p in range(pos):
        for i in range(d_model):
            angle = p / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
            table[p, i] = np.sin(angle) if i % 2 == 0 else np.cos(angle)
    return table

print('슝=3')

슝=3


In [13]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.depth     = d_model // num_heads
        self.d_model   = d_model
        self.W_q    = nn.Linear(d_model, d_model)
        self.W_k    = nn.Linear(d_model, d_model)
        self.W_v    = nn.Linear(d_model, d_model)
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(Q.size(-1))
        if mask is not None:
            scores = scores + (mask * -1e9)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V), attn

    def split_heads(self, x):
        b, s, _ = x.size()
        return x.view(b, s, self.num_heads, self.depth).permute(0, 2, 1, 3)

    def combine_heads(self, x):
        b, _, s, _ = x.size()
        return x.permute(0, 2, 1, 3).contiguous().view(b, s, self.d_model)

    def forward(self, Q, K, V, mask=None):
        out, attn = self.scaled_dot_product_attention(
            self.split_heads(self.W_q(Q)),
            self.split_heads(self.W_k(K)),
            self.split_heads(self.W_v(V)), mask)
        return self.linear(self.combine_heads(out)), attn

print('슝=3')

슝=3


In [14]:
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1  = nn.Linear(d_model, d_ff)
        self.fc2  = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn    = PoswiseFeedForwardNet(d_model, d_ff)
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.do     = nn.Dropout(dropout)

    def forward(self, x, mask):
        n = self.norm_1(x)
        out, attn = self.enc_self_attn(n, n, n, mask)
        out = self.do(out) + x
        residual = out
        out = self.do(self.ffn(self.norm_2(out))) + residual
        return out, attn


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.dec_self_attn = MultiHeadAttention(d_model, n_heads)
        self.enc_dec_attn  = MultiHeadAttention(d_model, n_heads)
        self.ffn    = PoswiseFeedForwardNet(d_model, d_ff)
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)
        self.do     = nn.Dropout(dropout)

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        n = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(n, n, n, mask=padding_mask)
        out = self.do(out) + x
        residual = out
        n = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(n, enc_out, enc_out, mask=dec_enc_mask)
        out = self.do(out) + residual
        residual = out
        out = self.do(self.ffn(self.norm_3(out))) + residual
        return out, dec_attn, dec_enc_attn

print('슝=3')

슝=3


In [15]:
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.layers     = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout)
                                         for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x, mask):
        attns = []
        for layer in self.layers:
            x, a = layer(x, mask)
            attns.append(a)
        return self.final_norm(x), attns


class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.layers     = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout)
                                         for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        dec_attns, dec_enc_attns = [], []
        for layer in self.layers:
            x, da, dea = layer(x, enc_out, dec_enc_mask, padding_mask)
            dec_attns.append(da)
            dec_enc_attns.append(dea)
        return self.final_norm(x), dec_attns, dec_enc_attns

print('슝=3')

슝=3


In [16]:
class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff,
                 vocab_size, pos_len, dropout=0.2):
        super().__init__()
        self.d_model = float(d_model)
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        pos_enc  = positional_encoding(pos_len, d_model)
        self.register_buffer('pos_encoding', torch.tensor(pos_enc, dtype=torch.float32))
        self.do      = nn.Dropout(dropout)
        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.fc      = nn.Linear(d_model, vocab_size)
        self.fc.weight = self.emb.weight

    def embedding(self, x):
        out = self.emb(x) * math.sqrt(self.d_model)
        return self.do(out + self.pos_encoding[:x.size(1)].unsqueeze(0))

    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        enc_out, enc_attns = self.encoder(self.embedding(enc_in), enc_mask)
        dec_out, dec_attns, dec_enc_attns = self.decoder(
            self.embedding(dec_in), enc_out, dec_enc_mask, dec_mask)
        return self.fc(dec_out), enc_attns, dec_attns, dec_enc_attns

print('슝=3')

슝=3


In [17]:
def generate_padding_mask(seq):
    return (seq == PAD_ID).unsqueeze(1).unsqueeze(2).float()

def generate_lookahead_mask(size):
    return torch.triu(torch.ones(size, size), diagonal=1)

def generate_masks(src, tgt):
    enc_mask     = generate_padding_mask(src)
    dec_enc_mask = generate_padding_mask(src)
    lookahead    = generate_lookahead_mask(tgt.size(1)).to(tgt.device)
    tgt_pad      = (tgt == PAD_ID).unsqueeze(1).unsqueeze(1).float()
    dec_mask     = torch.max(tgt_pad, lookahead.unsqueeze(0).unsqueeze(0))
    return enc_mask, dec_enc_mask, dec_mask

print('슝=3')

슝=3


In [18]:
class LearningRateScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, d_model, warmup_steps=5, last_epoch=-1):
        self.d_model      = d_model
        self.warmup_steps = warmup_steps
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        step = max(1, self.last_epoch)
        lr   = (self.d_model ** -0.5) * min(step ** -0.5, step * (self.warmup_steps ** -1.5))
        return [lr for _ in self.base_lrs]

d_model = 256
transformer = Transformer(
    n_layers=2, d_model=d_model, n_heads=8, d_ff=512,
    vocab_size=VOCAB_SIZE, pos_len=MAX_LEN + 10, dropout=0.1
).to(device)

optimizer    = torch.optim.Adam(transformer.parameters(), lr=1e-9, betas=(0.9, 0.98), eps=1e-9)
lr_scheduler = LearningRateScheduler(optimizer, d_model=d_model, warmup_steps=5)

print(f'파라미터 수: {sum(p.numel() for p in transformer.parameters()):,}')
print('슝=3')

파라미터 수: 4,692,800
슝=3


In [19]:
def loss_function(real, pred):
    loss_ = F.cross_entropy(
        pred.contiguous().view(-1, pred.size(-1)),
        real.contiguous().view(-1).to(device),
        reduction='none'
    ).view(real.size())
    mask = (real != PAD_ID).float().to(device)
    return (loss_ * mask).sum() / mask.sum()


def train_step(src, tgt, model, optimizer):
    model.train()
    optimizer.zero_grad()
    tgt_in = tgt[:, :-1]
    gold   = tgt[:, 1:]

    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)
    preds, *_ = model(
        src.to(device), tgt_in.to(device),
        enc_mask.to(device), dec_enc_mask.to(device), dec_mask.to(device))
    loss = loss_function(gold, preds)
    loss.backward()
    optimizer.step()
    return loss

print('슝=3')

슝=3


In [20]:
from tqdm.notebook import tqdm

EPOCHS = 30

for epoch in range(EPOCHS):
    total_loss, n = 0.0, len(train_dataloader)
    pbar = tqdm(total=n, desc=f'Epoch {epoch+1:2d}/{EPOCHS}')

    for src, tgt in train_dataloader:
        loss = train_step(src, tgt, transformer, optimizer)
        total_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}',
                         lr=f'{optimizer.param_groups[0]["lr"]:.2e}')
        pbar.update(1)

    lr_scheduler.step()
    pbar.close()
    print(f'Epoch {epoch+1:2d} -- avg loss: {total_loss/n:.4f}')

Epoch  1/30:   0%|          | 0/326 [00:00<?, ?it/s]

Epoch  1 -- avg loss: 12.5045


Epoch  2/30:   0%|          | 0/326 [00:00<?, ?it/s]

Epoch  2 -- avg loss: 5.1805


Epoch  3/30:   0%|          | 0/326 [00:00<?, ?it/s]

...

Epoch 28 -- avg loss: 1.7370


Epoch 29/30:   0%|          | 0/326 [00:00<?, ?it/s]

Epoch 29 -- avg loss: 1.6703


Epoch 30/30:   0%|          | 0/326 [00:00<?, ?it/s]

Epoch 30 -- avg loss: 1.6558


### **Step 7. 모델 평가하기**

챗봇이 올바른 대답을 하는지 테스트하고, BLEU Score로 평가합니다.

In [21]:
def predict(sentence, model, tokenizer, max_len=MAX_LEN):
    model.eval()
    sentence = preprocess_sentence(sentence)
    ids = tokenizer.encode_as_ids(sentence)
    ids = ids[:max_len] + [PAD_ID] * max(0, max_len - len(ids))

    src    = torch.tensor([ids], dtype=torch.long, device=device)
    output = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(max_len):
            enc_mask, dec_enc_mask, dec_mask = generate_masks(src, output)
            preds, *_ = model(src, output,
                              enc_mask.to(device),
                              dec_enc_mask.to(device),
                              dec_mask.to(device))
            pred_id = preds[0, -1].argmax(-1).item()
            if pred_id == EOS_ID:
                break
            output = torch.cat([output,
                                 torch.tensor([[pred_id]], device=device)], dim=1)

    pred_ids = output[0, 1:].tolist()
    return tokenizer.decode_ids(pred_ids)

In [22]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def calculate_bleu(reference, candidate):
    ref_tokens  = reference.split()
    cand_tokens = candidate.split()
    return sentence_bleu([ref_tokens], cand_tokens,
                         smoothing_function=SmoothingFunction().method1)

In [23]:
test_sentences = [
    '안녕하세요',
    '오늘 기분이 어때요?',
    '배가 고파요',
    '심심해요',
    '사랑해',
]

print('=== 챗봇 테스트 ===')
for q in test_sentences:
    ans = predict(q, transformer, tokenizer)
    print(f'Q: {q}')
    print(f'A: {ans}')
    print()

=== 챗봇 테스트 ===
Q: 안녕하세요
A: 감기 조심하세요 .

Q: 오늘 기분이 어때요?
A: 그 사람을 위해 에너지를 하나밖에 에너지를 했나 봐요 .

Q: 배가 고파요
A: 컨디션 조절 조절 조절 조절 조절 조절 조절 조절 조절 조절 하세요 .

Q: 심심해요
A: 그래도 상관없어요 . 칭찬해주고 싶네요 .

Q: 사랑해
A: 더 신경써 주세요 .



In [24]:
N_TEST = min(200, len(src_sents))
test_q = src_sents[-N_TEST:]
test_a = tgt_sents[-N_TEST:]

scores = []
for q, ref in tqdm(zip(test_q, test_a), total=N_TEST, desc='BLEU 평가'):
    pred  = predict(q, transformer, tokenizer)
    score = calculate_bleu(ref, pred)
    scores.append(score)

print(f'평균 BLEU: {sum(scores)/len(scores):.4f}')
print(f'최고 BLEU: {max(scores):.4f}')

BLEU 평가:   0%|          | 0/200 [00:00<?, ?it/s]

평균 BLEU: 0.0590
최고 BLEU: 1.0000


---
## 프로젝트 결과 분석 및 회고


![실행 결과 비교](result_combined.png)

### 실행 결과 비교

| | sub_ratio=0.2, N_AUG=1 | sub_ratio=0.2, N_AUG=3 | sub_ratio=0.4, N_AUG=1 | sub_ratio=0.4, N_AUG=3 |
|---|---|---|---|---|
| 평균 BLEU | 0.0503 | 0.0590 | **0.0546** | 0.0526 |
| 최고 BLEU | 0.1710 | 1.0000* | **0.4005** | 0.1699 |
| 증강 쌍 수 | ~14,928 | ~20,767 | 더 많음 | 더 많음 |
| 비고 | 맥락 불일치 多 | 반복 토큰 발생 | 가장 자연스러움 | 문맥 불일치 |

> *N_AUG=3의 최고 BLEU 1.0은 정답과 우연히 완전 일치한 케이스

**sub_ratio=0.4, N_AUG=1이 가장 좋은 품질**이었다. 치환 비율을 높이되 증강 횟수를 1회로 제한하는 것이 더 효과적이었다.

```
Q: 심심해요    → A: 제가 있잖아요.
Q: 사랑해      → A: 지금도 충분히 잘 하고 있어요.
```

### 발견된 문제점

**1. N_AUG=3에서 반복 토큰 생성**

- `배가 고파요` → `컨디션 조절 조절 조절 조절...`
- 증강 횟수가 늘수록 Q-A 쌍의 품질 편차가 커지고, 노이즈 데이터가 학습을 불안정하게 만든 것으로 보임
- 모델이 EOS 생성을 학습하지 못했을 때 직전 토큰을 반복하는 패턴으로 빠짐

**2. sub_ratio 증가가 항상 유리하지 않음**

- 0.2 → 0.4로 올렸을 때 N_AUG=1에서는 성능이 올랐으나(0.0503 → 0.0546), N_AUG=3에서는 오히려 내려감(0.0590 → 0.0526)
- 치환 비율이 높을수록 원문 의미가 변질되고, 반복 증강과 결합하면 노이즈가 누적됨

**3. 학습마다 결과 편차가 큼**

- 소형 모델(d=256) + 1만 쌍 규모에서 가중치 초기화 방향에 따라 로컬 미니마가 달라짐


### 개선 방향

| 항목 | 현재 | 개선안 | 우선순위 |
|---|---|---|---|
| 반복 토큰 | 없음 | repetition penalty 추가 | 높음 |
| 증강 설정 | sub_ratio=0.4, N_AUG=1 유지 | N_AUG 늘릴 경우 sub_ratio 낮추기 | 높음 |
| 모델 크기 | d=256, 2 layers | d=512, 4 layers | 중간 |
| 학습 에폭 | 30 | 50~100 (loss 수렴 확인) | 중간 |
| 평가 지표 | BLEU만 | BERTScore 병행 | 낮음 |


### 회고

- **증강 설정의 트레이드오프**: sub_ratio와 N_AUG를 동시에 높이면 데이터 양은 늘지만 품질이 떨어진다. 실험 결과 sub_ratio=0.4 + N_AUG=1 조합이 가장 효과적이었다.

- **SentencePiece BPE 교체가 핵심 개선**이었다. 형태소 분리 방식은 공백 정보를 잃어 출력 복원이 불가능했지만, BPE는 `decode_ids()` 한 번으로 자연스러운 문장이 복원된다.

- **환경 의존성 주의**: MeCab이 Windows에서 Visual Studio 없이 빌드 실패했다. 운영체제에 의존하는 라이브러리는 초기 환경 설정 단계에서 검증해야 한다.

- **LR 스케줄러 단위 주의**: 배치 단위가 아닌 에폭 단위로 `step()`을 호출하면 `warmup_steps` 설정 기준이 달라진다. 이론과 구현 사이의 간극을 직접 실험으로 확인했다.

- 챗봇에선 비슷한 의미도 사람이 평가하기엔 정답이기에 BLEU지표만을 믿을 수는 없다. 하이퍼 파라미터 튜닝이나 repetition penality를 사용한다면 품질이 올라갈 수 있을 것같다.